# Cell type phenotyping and annotation

Explore the full pancreas cohort using the updated Islet Explorer protein assignments. The same UMAP shows where cell identities, disease groups, and protein signals overlap. The heatmap and dotplots summarize the proteins that distinguish each phenotype.

Adapted from the [original PhenoCycler analysis notebook](https://github.com/smith6jt-cop/Phenocycler_Analysis/blob/ee170f13ebb892bc395bb289625df038ad0fadb1/notebooks/02_phenotyping.ipynb).

## 1. Load the processed PhenoCycler data

The November 2 protein hierarchy is the source for the November 13 native-interior correction and subsequent refinement. The cohort UMAP sample enriches uncommon cell types. Marker summaries and cell proportions use all admitted cells.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm
import anndata as ad
import scanpy as sc
from IPython.display import display

# From a separate checkout, set PHENOCYCLER_DATA to the folder containing the .h5ad files.
cwd = Path.cwd()
candidates = [Path(os.environ["PHENOCYCLER_DATA"])] if "PHENOCYCLER_DATA" in os.environ else [
    cwd / "analysis" / "phenocycler" / "scanpy", cwd / "phenocycler" / "scanpy",
    cwd / "scanpy", cwd.parent / "scanpy", cwd]
data = next((d for d in candidates if (d / "pancreas_islets.h5ad").is_file()), None)
if data is None:
    raise FileNotFoundError("Set PHENOCYCLER_DATA to the processed PhenoCycler data directory.")
sc.set_figure_params(dpi=100, facecolor="white", fontsize=10, frameon=False)
plt.rcParams.update({"axes.spines.top":False, "axes.spines.right":False})
DONOR = "6533"                      # Any donor listed in the manifest
DISEASE = "T1D"                     # ND, AAb+, or T1D
LEVEL = "primary"                    # primary, detail, immune_class, t_subtype, macrophage_profile
MARKERS = ["GCG","INS","SST","CD99","CD3e","CD20","CD68","CD31","EpCAM"]
PAIR = ("INS","SST")
group_order = ["ND","AAb+","T1D"]
group_ink = {"ND":"#23699b","AAb+":"#ab6c06","T1D":"#ae4643"}
counts = pd.read_parquet(data / "donor_counts.parquet").query("tissue == 'Pancreas'").copy()
evidence = pd.read_parquet(data / "donor_markers.parquet").query("tissue == 'Pancreas'").copy()
counts["donor"] = counts.donor.astype(str)
evidence["donor"] = evidence.donor.astype(str)
manifest = pd.read_csv(data / "pancreas_manifest.csv",dtype={"donor":str})
if DONOR not in set(manifest.donor) or DISEASE not in group_order:
    raise ValueError("Choose a donor and group from pancreas_manifest.csv")
atlas = sc.read_h5ad(data / "pancreas_umap.h5ad")
coords = np.asarray(atlas.obsm["X_umap"])
mapped = atlas[np.isfinite(coords).all(axis=1)].copy()
print(f"{len(manifest)} pancreas donors · {manifest.cells.sum():,} admitted cells · {mapped.n_obs:,} displayed on the joint UMAP")
print("Disease groups:", ", ".join(f"{g} ({n} donors)" for g,n in manifest.groupby("disease").size().items()))
type_order = ["Alpha","Beta","Delta","Endocrine (ambiguous subtype)","Endocrine (unresolved subtype)",
              "Exocrine","Immune","Blood vessel","Lymphatic","Nerve","Unresolved (no lineage evidence)"]
inks = {"Alpha":"#cf6c30", "Beta":"#287cac", "Delta":"#4c8b58", "Immune":"#965084",
        "Exocrine":"#c1b7a6", "Endocrine (ambiguous subtype)":"#bda559",
        "Endocrine (unresolved subtype)":"#99853f", "Blood vessel":"#b34b50",
        "Lymphatic":"#68a69a", "Nerve":"#5b558c", "Unresolved (no lineage evidence)":"#868686"}
for label in type_order:
    inks.setdefault(str(label), "#73777d")

## 2. Visualize cell types on the shared protein UMAP

All panels use the same embedding and limits. Select a disease group or donor above to inspect the same population on those coordinates.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 7.6))
limits = [(coords[np.isfinite(coords[:,k]), k].min(), coords[np.isfinite(coords[:,k]), k].max()) for k in (0,1)]
sc.pl.umap(mapped, color="disease", palette=group_ink, size=3, ax=axes[0,0],
           title="Disease group", legend_loc="none", show=False)
for view, ax, title in [(mapped, axes[0,1], "All donors"),
                       (mapped[mapped.obs.disease.astype(str) == DISEASE].copy(), axes[1,0], DISEASE),
                       (mapped[mapped.obs.donor.astype(str) == DONOR].copy(), axes[1,1], f"Donor {DONOR}")]:
    sc.pl.umap(view, color="primary", palette=inks, ax=ax, size=3 if view.n_obs>8000 else 8,
               title=f"{title} · {view.n_obs:,} cells", legend_loc="none", show=False)
for ax in axes.flat:
    ax.set(xlim=limits[0], ylim=limits[1], aspect="equal")
for ax, palette, labels, ncol in [(axes[0,0], group_ink, group_order, 3),
                                   (axes[1,0], inks, mapped.obs.primary.dropna().unique(), 3)]:
    handles=[Line2D([],[],marker="o",linestyle="",markersize=5,color=palette[v],label=v) for v in labels]
    if ax is axes[0,0]:
        ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(.50,-.01), ncol=ncol, frameon=False, fontsize=8)
    else:
        fig.legend(handles=handles, loc="lower center", ncol=ncol, frameon=False, fontsize=8)
fig.subplots_adjust(top=.95, bottom=.15, hspace=.25)
plt.show()

## 3. Marker expression on the UMAP

Colors show individual proteins on the calibrated 0–1 typing scale. Gray denotes missing measurements.

In [ ]:
show_markers = [m for m in MARKERS if m in mapped.var_names][:6]
sc.pl.umap(mapped, color=show_markers, layer="typing_scaled", use_raw=False, ncols=3,
           size=4, cmap="viridis", vmin=0, vmax=1, na_color="#cecece", wspace=.26, hspace=.32)

## 4. Compare protein signatures across phenotypes

The heatmap uses the mean scaled expression among measured cells, with equal weight per measured donor. The dotplots add the fraction at or above 0.10. A gray × marks missing measurements. The donor plot includes individually assessable markers. The disease plot uses cross-donor comparable measurements.

In [ ]:
# One row per phenotype; donor means are equally weighted for each measured marker.
keys = counts.loc[counts.level.eq(LEVEL), ["donor","label","comparable"]]
valid = evidence[evidence.level.eq(LEVEL)].merge(keys,on=["donor","label"],validate="many_to_one")
valid = valid.loc[valid.comparable & valid.n_measured.gt(0) & valid.marker.isin(MARKERS)]
order = [t for t in type_order if t in valid.label.unique()] + sorted(set(valid.label) - set(type_order))
features = [m for m in MARKERS if m in set(valid.marker)]
heat = valid.pivot_table(index="label",columns="marker",values="mean_scaled",aggfunc="mean",
                         observed=True).reindex(index=order,columns=features)
if len(heat) and features:
    hm = ad.AnnData(heat.to_numpy(),obs=pd.DataFrame({"phenotype":pd.Categorical(order,order)},index=order),
                    var=pd.DataFrame(index=features))
    cmap=plt.get_cmap("viridis").copy()
    cmap.set_bad("#d9d9d9")
    sc.pl.heatmap(hm,features,groupby="phenotype",show_gene_labels=True,swap_axes=True,
                  cmap=cmap,vmin=0,vmax=1,figsize=(max(9,len(heat)*.65),3.9),show=False)
    ax=plt.gcf().axes[0]
    ax.set_title("Mean protein expression across donors",fontsize=12,pad=9)
    plt.show()
else:
    print(f"No comparable marker values for {LEVEL}.")

In [ ]:
def protein_dotplot(group=None, donor_id=None):
    eligible = counts.loc[counts.level.eq(LEVEL), ["donor","disease","label","comparable","assessable"]]
    frame = evidence.loc[evidence.level.eq(LEVEL) & evidence.marker.isin(MARKERS)].merge(
        eligible, on=["donor","disease","label"], validate="many_to_one")
    frame = frame.loc[frame.donor.eq(donor_id) if donor_id is not None else frame.disease.eq(group)].copy()
    markers = [m for m in MARKERS if m in evidence.marker.unique()]
    reportable = frame.assessable if donor_id is not None else frame.comparable
    frame = frame.loc[reportable & frame.n_measured.gt(0)]
    if frame.empty:
        print(f"No measured cells at {LEVEL} for this selection.")
        return
    labels = sorted(frame.label.unique(), key=lambda s: (type_order.index(s) if s in type_order else len(type_order), s))
    def matrix(column):
        return frame.pivot_table(index="label",columns="marker",values=column,aggfunc="mean",
                                 observed=True).reindex(index=labels,columns=markers)
    mean, fraction = matrix("mean_scaled"), matrix("positive_fraction")
    # Scanpy accepts precomputed sizes and colors. These summaries preserve cell and donor denominators.
    summary = ad.AnnData(mean.to_numpy(), obs=pd.DataFrame({"cell_type": pd.Categorical(labels,labels)},index=labels),
                         var=pd.DataFrame(index=markers))
    scope = f"Donor {donor_id}" if donor_id is not None else f"{group} · {frame.donor.nunique()} donors"
    plot = sc.pl.DotPlot(summary, markers, groupby="cell_type", dot_color_df=mean,
                         dot_size_df=fraction, vmin=0, vmax=1, title=f"{scope} · {LEVEL}",
                         figsize=(8.5, max(3.6, len(labels)*.28+1.8)))
    plot.style(cmap="viridis", dot_min=0, dot_max=1, largest_dot=150).legend(
        colorbar_title="Mean scaled protein", size_title="Positive cells (%)")
    plot.make_figure()
    ax = plot.get_axes()["mainplot_ax"]
    y, x = np.where(fraction.isna().to_numpy())
    ax.scatter(x+.5, y+.5, marker="x", s=16, linewidth=.8, color="#747474")
    plt.show()


protein_dotplot(group=DISEASE)
protein_dotplot(donor_id=DONOR)

## 5. Explore marker co-expression

Each donor contributes 4,000 randomly selected cells. The panels show the measured cells in the disease group and selected donor. Axes use aligned protein intensity after log(1 + intensity). The scale retains differences above the ceiling used for typing.

In [ ]:
scatter_cells = sc.read_h5ad(data / "pancreas_random.h5ad")
pair = scatter_cells[:, list(PAIR)].copy()
raw = np.asarray(pair.X)
good = np.isfinite(raw).all(axis=1) & (raw >= 0).all(axis=1)
pair = pair[good].copy()
pair.X = np.log1p(np.asarray(pair.X))
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.1), sharex=True, sharey=True)
for view, ax, title in [(pair[pair.obs.disease.astype(str)==DISEASE].copy(), axes[0], DISEASE),
                       (pair[pair.obs.donor.astype(str)==DONOR].copy(), axes[1], f"Donor {DONOR}")]:
    if view.n_obs:
        sc.pl.scatter(view, x=PAIR[0], y=PAIR[1], color="primary", palette=[inks[c] for c in view.obs.primary.cat.categories], size=10 if view.n_obs < 8000 else 4,
                      alpha=.72, ax=ax, show=False, use_raw=False, legend_loc="none", frameon=True)
    ax.grid(False)
    ax.set(title=f"{title} · {view.n_obs:,} measured cells", xlabel=f"log1p({PAIR[0]})", ylabel=f"log1p({PAIR[1]})")
used = set(pair.obs.loc[pair.obs.disease.astype(str).eq(DISEASE) | pair.obs.donor.astype(str).eq(DONOR),"primary"].dropna())
handles=[Line2D([],[],marker="o",linestyle="",markersize=5,color=inks[label],label=label)
         for label in type_order if label in used]
fig.legend(handles=handles,loc="lower center",ncol=4,frameon=False,fontsize=8)
fig.subplots_adjust(bottom=.29, wspace=.22)
plt.show()

Full donor files contain all 68 marker columns with availability and compartment metadata, both expression scales, original and updated cell types, and native coordinates. Change the field in the optional example to use other Scanpy functions.

In [ ]:
donor = sc.read_h5ad(data / "pancreas" / f"{DONOR}.h5ad", backed="r")
print(donor)
display(donor.obs["celltype"].value_counts().rename("admitted cells").to_frame())
# For your own Scanpy plot, load just the desired cells/markers from the backed object:
# small = donor[:10000, ["GCG","INS","SST"]].to_memory()
# sc.pl.violin(small, ["GCG","INS","SST"], groupby="celltype", layer="typing_scaled")

Continue with [spatial analysis and donor trajectories](03_spatial_analysis.ipynb) or [group comparisons and proportions](04_group_comparisons.ipynb).